# DB9_Ultimate trên Colab (A100/H100/L4)
1. Runtime → Change runtime type → GPU (A100/H100 nếu có).
2. Điền `GITHUB_TOKEN` (repo private) và `HF_TOKEN` (model FLUX.2 của Black Forest Labs cần đồng ý license trên Hugging Face).
3. Chạy lần lượt các ô. Ảnh ra nằm trong `ComfyUI/output/DB9U/` (DB9U Save tự nhận cloud).

In [ ]:
GITHUB_TOKEN = ""   # token GitHub có quyền đọc repo bemapa3/DB9_Ultimate
HF_TOKEN = ""       # token Hugging Face (cần cho FLUX.2 klein)
MODEL = "qwen21"    # "qwen21" hoặc "flux2_klein_9b"


In [ ]:
%cd /content
!git clone -q https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI
!pip -q install -r requirements.txt opencv-python-headless
!git clone -q https://$GITHUB_TOKEN@github.com/bemapa3/DB9_Ultimate custom_nodes/DB9_Ultimate
!git clone -q https://github.com/b2renger/ComfyUI-QwenImage21-FunControlNet custom_nodes/ComfyUI-QwenImage21-FunControlNet


In [ ]:
import os, subprocess
def get(url, folder):
    os.makedirs(folder, exist_ok=True)
    name = url.split("/")[-1]
    if not os.path.exists(f"{folder}/{name}"):
        h = ["--header", f"Authorization: Bearer {HF_TOKEN}"] if HF_TOKEN else []
        subprocess.run(["wget", "-q", "-c", *h, url, "-O", f"{folder}/{name}"], check=True)
M = "/content/ComfyUI/models"
if MODEL == "qwen21":  # nguồn: note trong workflow mẫu chính thức Comfy-Org (Qwen-Image 2.1)
    get("https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/diffusion_models/qwen_image_2.1_int8_convrot.safetensors", f"{M}/diffusion_models")
    get("https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/text_encoders/qwen3vl_8b_int8_convrot.safetensors", f"{M}/text_encoders")
    get("https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/vae/qwen_image_2.1_vae_bf16.safetensors", f"{M}/vae")
else:  # nguồn: workflow mẫu chính thức image_flux2_klein_image_edit_9b_distilled.json
    get("https://huggingface.co/black-forest-labs/FLUX.2-klein-9b-fp8/resolve/main/flux-2-klein-9b-fp8.safetensors", f"{M}/diffusion_models")
    get("https://huggingface.co/Comfy-Org/flux2-klein-9B/resolve/main/split_files/text_encoders/qwen_3_8b_fp8mixed.safetensors", f"{M}/text_encoders")
    get("https://huggingface.co/black-forest-labs/FLUX.2-small-decoder/resolve/main/full_encoder_small_decoder.safetensors", f"{M}/vae")
    print("Workflow Flux: đổi VAE sang full_encoder_small_decoder.safetensors")
print("xong")


In [ ]:
# Chạy test nhanh trước
!python custom_nodes/DB9_Ultimate/tests/test_core.py | tail -1
!python custom_nodes/DB9_Ultimate/tests/test_engine_mock.py | tail -1


In [ ]:
# Mở ComfyUI qua link public (cloudflared)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
import subprocess, threading, time, re
subprocess.Popen(["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"], cwd="/content/ComfyUI")
time.sleep(15)
p = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"], stderr=subprocess.PIPE, text=True)
for line in p.stderr:
    m = re.search(r"https://[\w-]+\.trycloudflare\.com", line)
    if m:
        print("Mở ComfyUI:", m.group(0)); break


Kéo file `custom_nodes/DB9_Ultimate/workflows/DB9U_*.json` vào ComfyUI. Ảnh lưu ở `ComfyUI/output/DB9U/` — tải về bằng thanh Files bên trái.